# HoloBox Digital Hologram Processing

This notebook provides comprehensive hologram processing capabilities for the HoloBox system using xeus-python kernel.

## Features
- Fresnel propagation algorithms
- Auto-focus capabilities
- Interactive visualization
- Real-time parameter adjustment
- Camera integration ready

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
import ipywidgets as widgets
from IPython.display import display, clear_output
import requests
import io
from PIL import Image

print('HoloBox Processing Environment Initialized!')
print('Available kernels: xeus-python')
print('Scientific stack: NumPy, SciPy, Matplotlib loaded')

## Core Hologram Processing Functions

In [ ]:
class HologramProcessor:
    """Complete hologram processing pipeline for HoloBox."""
    
    def __init__(self, wavelength=440e-9, pixel_size=1.4e-6):
        self.wavelength = wavelength
        self.pixel_size = pixel_size
        
    def fresnel_propagate(self, field, distance):
        """
        Fresnel propagation of optical field.
        
        Parameters:
        field: 2D complex array - input optical field
        distance: float - propagation distance in meters
        
        Returns:
        2D complex array - propagated field
        """
        ny, nx = field.shape
        
        # Frequency coordinates
        fx = np.fft.fftfreq(nx, self.pixel_size)
        fy = np.fft.fftfreq(ny, self.pixel_size)
        FX, FY = np.meshgrid(fx, fy)
        
        # Fresnel transfer function
        H = np.exp(1j * np.pi * self.wavelength * distance * (FX**2 + FY**2))
        
        # Propagate
        field_fft = np.fft.fft2(field)
        propagated_fft = field_fft * H
        propagated_field = np.fft.ifft2(propagated_fft)
        
        return propagated_field
    
    def reconstruct_hologram(self, hologram, distance=0.005):
        """
        Complete hologram reconstruction pipeline.
        
        Parameters:
        hologram: 2D array - input hologram intensity
        distance: float - reconstruction distance in meters
        
        Returns:
        dict with reconstruction results
        """
        # Normalize hologram
        hologram_normalized = hologram / np.max(hologram)
        
        # Convert to complex field (plane wave reference assumption)
        field = np.sqrt(hologram_normalized).astype(complex)
        
        # Propagate to reconstruction plane
        reconstructed = self.fresnel_propagate(field, distance)
        
        # Calculate results
        intensity = np.abs(reconstructed)**2
        phase = np.angle(reconstructed)
        amplitude = np.abs(reconstructed)
        
        return {
            'intensity': intensity,
            'phase': phase,
            'amplitude': amplitude,
            'complex_field': reconstructed
        }
    
    def find_optimal_distance(self, hologram, distance_range=(0.001, 0.010), num_steps=20):
        """
        Find optimal reconstruction distance using focus metrics.
        
        Parameters:
        hologram: 2D array - input hologram
        distance_range: tuple - (min_distance, max_distance) in meters
        num_steps: int - number of distances to test
        
        Returns:
        dict with focus analysis results
        """
        distances = np.linspace(distance_range[0], distance_range[1], num_steps)
        focus_metrics = []
        
        for dist in distances:
            result = self.reconstruct_hologram(hologram, dist)
            
            # Calculate focus metric (variance of intensity)
            focus_metric = np.var(result['intensity'])
            focus_metrics.append(focus_metric)
        
        # Find optimal distance
        optimal_idx = np.argmax(focus_metrics)
        optimal_distance = distances[optimal_idx]
        
        return {
            'distances': distances,
            'focus_metrics': np.array(focus_metrics),
            'optimal_distance': optimal_distance,
            'optimal_metric': focus_metrics[optimal_idx]
        }
    
    def generate_sample_hologram(self, size=512):
        """
        Generate a sample hologram for testing and demonstration.
        
        Parameters:
        size: int - image size (size x size pixels)
        
        Returns:
        2D array - sample hologram
        """
        # Create coordinate arrays
        x = np.arange(size) * self.pixel_size
        y = np.arange(size) * self.pixel_size
        X, Y = np.meshgrid(x - x.mean(), y - y.mean())
        
        # Create object field with multiple point sources
        object_field = np.zeros((size, size), dtype=complex)
        
        # Point source parameters
        positions = [(-100e-6, -50e-6), (80e-6, 30e-6), (0, 100e-6), (-60e-6, 70e-6)]
        distances = [3e-3, 4e-3, 5e-3, 6e-3]
        amplitudes = [1.0, 0.8, 0.6, 0.7]
        
        for (px, py), dist, amp in zip(positions, distances, amplitudes):
            # Spherical wave from point source
            r = np.sqrt((X - px)**2 + (Y - py)**2 + dist**2)
            k = 2 * np.pi / self.wavelength
            point_wave = amp * np.exp(1j * k * r) / r
            object_field += point_wave
        
        # Reference wave (plane wave)
        reference = np.ones((size, size), dtype=complex)
        
        # Interference pattern (hologram)
        total_field = object_field + reference
        hologram = np.abs(total_field)**2
        
        # Add realistic noise
        noise_level = 0.02 * np.max(hologram)
        noise = np.random.normal(0, noise_level, hologram.shape)
        hologram += noise
        
        return np.maximum(hologram, 0)  # Ensure non-negative

# Initialize processor
processor = HologramProcessor()
print("HologramProcessor initialized with:")
print(f"  Wavelength: {processor.wavelength*1e9:.0f} nm")
print(f"  Pixel size: {processor.pixel_size*1e6:.1f} µm")

## Interactive Hologram Processing Demo

In [ ]:
# Generate sample hologram
sample_hologram = processor.generate_sample_hologram(size=512)

# Interactive widgets for parameter control
wavelength_slider = widgets.FloatSlider(
    value=440e-9, min=400e-9, max=700e-9, step=10e-9,
    description='Wavelength (m):', style={'description_width': 'initial'},
    readout_format='.0e'
)

pixel_size_slider = widgets.FloatSlider(
    value=1.4e-6, min=0.5e-6, max=5.0e-6, step=0.1e-6,
    description='Pixel Size (m):', style={'description_width': 'initial'},
    readout_format='.1e'
)

distance_slider = widgets.FloatSlider(
    value=0.005, min=0.001, max=0.020, step=0.0005,
    description='Distance (m):', style={'description_width': 'initial'},
    readout_format='.3f'
)

regenerate_button = widgets.Button(
    description='Generate New Hologram',
    button_style='info'
)

autofocus_button = widgets.Button(
    description='Auto Focus',
    button_style='success'
)

# Output widget for plots
output = widgets.Output()

def update_reconstruction(change=None):
    """Update reconstruction when parameters change."""
    global sample_hologram
    
    with output:
        clear_output(wait=True)
        
        # Update processor parameters
        processor.wavelength = wavelength_slider.value
        processor.pixel_size = pixel_size_slider.value
        
        # Reconstruct hologram
        results = processor.reconstruct_hologram(sample_hologram, distance_slider.value)
        
        # Create visualization
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        
        # Original hologram
        im1 = axes[0,0].imshow(sample_hologram, cmap='gray')
        axes[0,0].set_title('Original Hologram')
        axes[0,0].axis('off')
        plt.colorbar(im1, ax=axes[0,0], shrink=0.8)
        
        # Reconstructed intensity
        im2 = axes[0,1].imshow(results['intensity'], cmap='hot')
        axes[0,1].set_title('Reconstructed Intensity')
        axes[0,1].axis('off')
        plt.colorbar(im2, ax=axes[0,1], shrink=0.8)
        
        # Phase
        im3 = axes[1,0].imshow(results['phase'], cmap='hsv')
        axes[1,0].set_title('Phase')
        axes[1,0].axis('off')
        plt.colorbar(im3, ax=axes[1,0], shrink=0.8)
        
        # Amplitude
        im4 = axes[1,1].imshow(results['amplitude'], cmap='viridis')
        axes[1,1].set_title('Amplitude')
        axes[1,1].axis('off')
        plt.colorbar(im4, ax=axes[1,1], shrink=0.8)
        
        plt.tight_layout()
        plt.show()
        
        # Print reconstruction info
        print(f"Reconstruction Parameters:")
        print(f"  Wavelength: {processor.wavelength*1e9:.0f} nm")
        print(f"  Pixel size: {processor.pixel_size*1e6:.1f} µm")
        print(f"  Distance: {distance_slider.value*1000:.1f} mm")
        print(f"  Intensity range: {results['intensity'].min():.2e} - {results['intensity'].max():.2e}")

def regenerate_hologram(button):
    """Generate new sample hologram."""
    global sample_hologram
    
    # Update processor parameters
    processor.wavelength = wavelength_slider.value
    processor.pixel_size = pixel_size_slider.value
    
    sample_hologram = processor.generate_sample_hologram()
    update_reconstruction()

def run_autofocus(button):
    """Run autofocus algorithm."""
    global sample_hologram
    
    with output:
        print("Running autofocus...")
    
    # Update processor parameters
    processor.wavelength = wavelength_slider.value
    processor.pixel_size = pixel_size_slider.value
    
    # Run autofocus
    focus_result = processor.find_optimal_distance(sample_hologram)
    
    # Update distance slider
    distance_slider.value = focus_result['optimal_distance']
    
    with output:
        print(f"Optimal distance found: {focus_result['optimal_distance']*1000:.2f} mm")
        
        # Plot focus curve
        plt.figure(figsize=(8, 4))
        plt.plot(focus_result['distances']*1000, focus_result['focus_metrics'], 'b-o')
        plt.axvline(focus_result['optimal_distance']*1000, color='r', linestyle='--', 
                   label=f'Optimal: {focus_result["optimal_distance"]*1000:.2f} mm')
        plt.xlabel('Distance (mm)')
        plt.ylabel('Focus Metric')
        plt.title('Autofocus Analysis')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()

# Connect widgets
wavelength_slider.observe(update_reconstruction, names='value')
pixel_size_slider.observe(update_reconstruction, names='value')
distance_slider.observe(update_reconstruction, names='value')
regenerate_button.on_click(regenerate_hologram)
autofocus_button.on_click(run_autofocus)

# Display interface
controls = widgets.VBox([
    widgets.HTML("<h3>Hologram Processing Controls</h3>"),
    wavelength_slider,
    pixel_size_slider, 
    distance_slider,
    widgets.HBox([regenerate_button, autofocus_button])
])

display(controls)
display(output)

# Initial reconstruction
update_reconstruction()

## HoloBox Camera Integration

The following cell demonstrates how to integrate with the HoloBox camera system for live hologram processing.

In [ ]:
def capture_and_process_hologram(camera_url="http://localhost:8000"):
    """
    Capture hologram from HoloBox camera and process it.
    
    Parameters:
    camera_url: str - Base URL of HoloBox camera API
    
    Returns:
    dict - Processing results
    """
    try:
        # Capture image from camera
        response = requests.get(f"{camera_url}/api/capture", timeout=10)
        
        if response.status_code == 200:
            # Convert response to image
            image_data = response.content
            image = Image.open(io.BytesIO(image_data))
            
            # Convert to numpy array (grayscale)
            hologram = np.array(image.convert('L'), dtype=float)
            
            # Process hologram
            results = processor.reconstruct_hologram(hologram)
            
            # Display results
            fig, axes = plt.subplots(1, 3, figsize=(15, 5))
            
            axes[0].imshow(hologram, cmap='gray')
            axes[0].set_title('Live Hologram')
            axes[0].axis('off')
            
            axes[1].imshow(results['intensity'], cmap='hot')
            axes[1].set_title('Reconstruction')
            axes[1].axis('off')
            
            axes[2].imshow(results['phase'], cmap='hsv')
            axes[2].set_title('Phase')
            axes[2].axis('off')
            
            plt.tight_layout()
            plt.show()
            
            print(f"Live hologram processed successfully!")
            print(f"Image size: {hologram.shape}")
            
            return results
            
        else:
            print(f"Failed to capture image. Status code: {response.status_code}")
            return None
            
    except requests.exceptions.RequestException as e:
        print(f"Camera connection error: {e}")
        print("Make sure HoloBox camera server is running at http://localhost:8000")
        return None
    except Exception as e:
        print(f"Processing error: {e}")
        return None

# Create button for live capture
live_capture_button = widgets.Button(
    description='Capture & Process Live',
    button_style='warning',
    icon='camera'
)

def capture_live(button):
    """Capture and process live hologram."""
    with output:
        clear_output(wait=True)
        print("Capturing live hologram...")
        result = capture_and_process_hologram()
        
        if result:
            print("Live processing completed successfully!")
        else:
            print("Live capture failed. Using sample hologram instead.")
            update_reconstruction()

live_capture_button.on_click(capture_live)

display(widgets.VBox([
    widgets.HTML("<h3>Live Camera Integration</h3>"),
    widgets.HTML("<p>Connect to HoloBox camera for real-time hologram processing.</p>"),
    live_capture_button
]))

## Advanced Processing Examples

The following cells demonstrate advanced hologram processing techniques.

In [ ]:
# Batch processing for focus optimization
def batch_focus_analysis(hologram, distance_range=(0.001, 0.020), num_distances=50):
    """
    Perform detailed focus analysis across multiple distances.
    """
    distances = np.linspace(distance_range[0], distance_range[1], num_distances)
    focus_metrics = []
    intensity_profiles = []
    
    print(f"Analyzing {num_distances} reconstruction distances...")
    
    for i, dist in enumerate(distances):
        if i % 10 == 0:
            print(f"  Progress: {i}/{num_distances}")
            
        result = processor.reconstruct_hologram(hologram, dist)
        
        # Multiple focus metrics
        variance_metric = np.var(result['intensity'])
        gradient_metric = np.mean(np.abs(np.gradient(result['intensity'])))
        
        focus_metrics.append({
            'distance': dist,
            'variance': variance_metric,
            'gradient': gradient_metric
        })
        
        # Store central line profile
        center_line = result['intensity'][result['intensity'].shape[0]//2, :]
        intensity_profiles.append(center_line)
    
    return {
        'distances': distances,
        'metrics': focus_metrics,
        'profiles': np.array(intensity_profiles)
    }

# Run batch analysis
print("Running comprehensive focus analysis...")
batch_results = batch_focus_analysis(sample_hologram)

# Visualize results
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Focus metrics
variances = [m['variance'] for m in batch_results['metrics']]
gradients = [m['gradient'] for m in batch_results['metrics']]

axes[0,0].plot(batch_results['distances']*1000, variances, 'b-', label='Variance')
axes[0,0].set_xlabel('Distance (mm)')
axes[0,0].set_ylabel('Intensity Variance')
axes[0,0].set_title('Focus Metric: Variance')
axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(batch_results['distances']*1000, gradients, 'r-', label='Gradient')
axes[0,1].set_xlabel('Distance (mm)')
axes[0,1].set_ylabel('Mean Gradient')
axes[0,1].set_title('Focus Metric: Gradient')
axes[0,1].grid(True, alpha=0.3)

# Intensity profiles vs distance
im1 = axes[1,0].imshow(batch_results['profiles'], aspect='auto', 
                      extent=[0, batch_results['profiles'].shape[1], 
                             batch_results['distances'][-1]*1000, 
                             batch_results['distances'][0]*1000],
                      cmap='hot')
axes[1,0].set_xlabel('Pixel Position')
axes[1,0].set_ylabel('Distance (mm)')
axes[1,0].set_title('Intensity Profiles vs Distance')
plt.colorbar(im1, ax=axes[1,0])

# Optimal distances
opt_var_idx = np.argmax(variances)
opt_grad_idx = np.argmax(gradients)

axes[1,1].plot(batch_results['distances']*1000, variances, 'b-', alpha=0.7, label='Variance')
axes[1,1].plot(batch_results['distances']*1000, np.array(gradients)/np.max(gradients)*np.max(variances), 
              'r-', alpha=0.7, label='Gradient (normalized)')
axes[1,1].axvline(batch_results['distances'][opt_var_idx]*1000, color='blue', linestyle='--',
                 label=f'Var optimum: {batch_results["distances"][opt_var_idx]*1000:.1f}mm')
axes[1,1].axvline(batch_results['distances'][opt_grad_idx]*1000, color='red', linestyle='--',
                 label=f'Grad optimum: {batch_results["distances"][opt_grad_idx]*1000:.1f}mm')
axes[1,1].set_xlabel('Distance (mm)')
axes[1,1].set_ylabel('Focus Metric')
axes[1,1].set_title('Comparison of Focus Metrics')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Analysis complete!")
print(f"Variance-based optimal distance: {batch_results['distances'][opt_var_idx]*1000:.2f} mm")
print(f"Gradient-based optimal distance: {batch_results['distances'][opt_grad_idx]*1000:.2f} mm")

## Export and Save Functions

In [ ]:
def save_results(hologram, results, filename_prefix="hologram_processing"):
    """
    Save processing results to files.
    
    Parameters:
    hologram: 2D array - original hologram
    results: dict - processing results
    filename_prefix: str - prefix for saved files
    """
    import json
    from datetime import datetime
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Save arrays as numpy files
    np.save(f"{filename_prefix}_{timestamp}_hologram.npy", hologram)
    np.save(f"{filename_prefix}_{timestamp}_intensity.npy", results['intensity'])
    np.save(f"{filename_prefix}_{timestamp}_phase.npy", results['phase'])
    np.save(f"{filename_prefix}_{timestamp}_amplitude.npy", results['amplitude'])
    
    # Save parameters
    params = {
        'timestamp': timestamp,
        'wavelength': processor.wavelength,
        'pixel_size': processor.pixel_size,
        'distance': distance_slider.value,
        'hologram_shape': hologram.shape,
        'intensity_range': [float(results['intensity'].min()), float(results['intensity'].max())],
        'phase_range': [float(results['phase'].min()), float(results['phase'].max())]
    }
    
    with open(f"{filename_prefix}_{timestamp}_params.json", 'w') as f:
        json.dump(params, f, indent=2)
    
    print(f"Results saved with timestamp: {timestamp}")
    return timestamp

# Example usage
current_results = processor.reconstruct_hologram(sample_hologram, distance_slider.value)
save_timestamp = save_results(sample_hologram, current_results)

print("\nProcessing session summary:")
print(f"Total cells executed: Multiple")
print(f"Current parameters:")
print(f"  Wavelength: {processor.wavelength*1e9:.0f} nm")
print(f"  Pixel size: {processor.pixel_size*1e6:.1f} µm") 
print(f"  Distance: {distance_slider.value*1000:.1f} mm")
print(f"Files saved with timestamp: {save_timestamp}")